# Sistema de Recomendação Visual Completo (Versão Integrada)

Este notebook integra todas as funcionalidades discutidas: configuração automática do Kaggle, extração de características visuais (forma, cor e textura) usando a ResNet50, visualizações de agrupamento (t-SNE), métricas de similaridade e filtros específicos por gênero (Masculino/Feminino).

## 1. Configuração e Download do Dataset
Faça o upload do seu arquivo `kaggle.json` quando solicitado para baixar o dataset.

In [ ]:
!pip install -q kaggle seaborn pandas
from google.colab import files
import os
import pandas as pd

if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("Por favor, faça o upload do seu arquivo kaggle.json:")
    files.upload()
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d paramaggarwal/fashion-product-images-small
!unzip -q fashion-product-images-small.zip -d dataset

## 2. Inicialização do Modelo e Funções de Extração
Extraímos o 'DNA visual' (vetores numéricos) da penúltima camada da ResNet50.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.preprocessing import image
from sklearn.neighbors import NearestNeighbors
from sklearn.manifold import TSNE

model = ResNet50(weights='imagenet', include_top=False, pooling='avg')

def extract_features(img_path, model):
    try:
        img = image.load_img(img_path, target_size=(224, 224))
        img_array = image.img_to_array(img)
        expanded_img_array = np.expand_dims(img_array, axis=0)
        preprocessed_img = preprocess_input(expanded_img_array)
        features = model.predict(preprocessed_img, verbose=0)
        return features.flatten() / np.linalg.norm(features.flatten())
    except:
        return None

# Carregando metadados para suporte a filtros
df = pd.read_csv('dataset/styles.csv', on_bad_lines='skip')
df['id'] = df['id'].astype(str)
image_dir = 'dataset/images'

## 3. Processamento do Dataset
Extraímos as características visuais para o banco de dados de recomendação.

In [ ]:
features_list = []
valid_image_paths = []
valid_metadata = []

print("Processando imagens e extraindo características visuais...")
for index, row in df.head(1000).iterrows():
    img_path = os.path.join(image_dir, row['id'] + '.jpg')
    if os.path.exists(img_path):
        feat = extract_features(img_path, model)
        if feat is not None:
            features_list.append(feat)
            valid_image_paths.append(img_path)
            valid_metadata.append(row)

features_list = np.array(features_list)
valid_metadata = pd.DataFrame(valid_metadata)
print(f"Sucesso: {len(features_list)} imagens prontas.")

## 4. Visualizações de Validação (t-SNE e Confiança)
Aqui demonstramos visualmente como a rede agrupa os itens e o nível de certeza das recomendações.

In [ ]:
def plot_tsne(features):
    tsne = TSNE(n_components=2, random_state=42)
    res = tsne.fit_transform(features)
    plt.figure(figsize=(10, 6))
    plt.scatter(res[:, 0], res[:, 1], alpha=0.5, c='blue', edgecolors='white')
    plt.title("Visualização t-SNE: Agrupamento por Aparência Física")
    plt.show()

plot_tsne(features_list)

def recommend_with_scenarios(img_id, gender_filter=None):
    # Encontrar a imagem base
    query_idx = valid_metadata[valid_metadata['id'] == str(img_id)].index[0]
    query_feat = features_list[query_idx]
    
    # Filtrar espaço de busca
    if gender_filter:
        indices = np.where(valid_metadata['gender'] == gender_filter)[0]
    else:
        indices = np.arange(len(features_list))
        
    search_space = features_list[indices]
    
    knn = NearestNeighbors(n_neighbors=6, metric='cosine')
    knn.fit(search_space)
    dist, idx_search = knn.kneighbors([query_feat])
    
    # Exibir resultados
    plt.figure(figsize=(15, 5))
    for i, pos in enumerate(idx_search[0]):
        real_pos = indices[pos]
        plt.subplot(1, 6, i+1)
        plt.imshow(mpimg.imread(valid_image_paths[real_pos]))
        plt.title("Busca" if i==0 else f"Conf: {1-dist[0][i]:.2f}")
        plt.axis('off')
    plt.show()

    # Gráfico de Confiança
    scores = [1 - d for d in dist[0][1:]]
    plt.figure(figsize=(8, 3))
    sns.barplot(x=scores, y=[f"Rec {i+1}" for i in range(5)], palette='magma')
    plt.xlim(0, 1)
    plt.title(f"Nível de Similaridade (Cenário: {gender_filter if gender_filter else 'Geral'})")
    plt.show()

# Executando exemplos
sample_id = valid_metadata.iloc[0]['id']
print("--- CENÁRIO MASCULINO ---")
recommend_with_scenarios(sample_id, gender_filter='Men')
print("\n--- CENÁRIO FEMININO ---")
recommend_with_scenarios(sample_id, gender_filter='Women')